# 03 - Train on a Kaggle GPU
1. On Kaggle, create a new notebook and upload this file (**File -> Import Notebook**).
2. In the right panel: **Add Input -> Competitions -> APTOS 2019 Blindness Detection**. Then open **Settings -> Accelerator -> GPU** and switch **Internet** on.
3. Replace `YOUR-USERNAME` below and run the cells in order.
4. Download `results.zip` at the end and unzip it into your local project folder.

In [ ]:
!git clone https://github.com/YOUR-USERNAME/dr-detection.git
%cd dr-detection
!pip install -q imagehash

Link the Kaggle dataset into `data/` (the input folder is read-only, so a link avoids copying 9 GB).

In [ ]:
import glob, os
csv = glob.glob("/kaggle/input/**/train.csv", recursive=True)[0]
src = os.path.dirname(csv)
print("Dataset found at", src)
os.makedirs("data", exist_ok=True)
for name in ("train.csv", "train_images"):
    if not os.path.exists(f"data/{name}"):
        os.symlink(f"{src}/{name}", f"data/{name}")
!ls data

In [ ]:
import tensorflow as tf
print("GPUs:", tf.config.list_physical_devices("GPU"))

In [ ]:
!python -m src.train --quick

## Main model and experiments
Run the cells you need. Each run takes roughly 15-40 minutes on a Kaggle GPU.

In [ ]:
!python -m src.train --backbone efficientnetb0 --exp-name effb0_full_weights

In [ ]:
!python -m src.train --backbone resnet50    --exp-name resnet50_full_weights
!python -m src.train --backbone mobilenetv2 --exp-name mobilenetv2_full_weights

In [ ]:
!python -m src.train --preprocess raw   --exp-name effb0_raw_weights
!python -m src.train --preprocess clahe --exp-name effb0_clahe_weights

In [ ]:
!python -m src.train --balance oversample --exp-name effb0_full_oversample
!python -m src.train --balance focal      --exp-name effb0_full_focal

In [ ]:
!python -m src.evaluate --exp-name effb0_full_weights --tta --benchmark
!python -m src.gradcam  --exp-name effb0_full_weights --n 8

## Package the results
Includes model weights, all figures, histories and the experiment log. The dataset is not included.

In [ ]:
!zip -qr /kaggle/working/results.zip models outputs data/splits
print("Download results.zip from the Output panel (right side).")